In [6]:
"""
Continuous Health Monitoring: LSTM Sequence Modeling (MVP)
==========================================================
Transforms tabular physiological epochs into 3D sequential tensors
and trains an LSTM network to detect anomalies based on temporal trajectories.

Run this in your Jupyter Notebook / Conda Environment.
Requires: pip install pandas numpy scikit-learn tensorflow
"""

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# --- Configuration & Paths ---
# Using raw strings as requested for local Windows paths
EPOCHS_PATH = r"C:\Users\Akshat - Personal\Visual Studio Code\Sensera\sensera-poc\model_dev_v1\health_epochs.csv"
USERS_PATH = r"C:\Users\Akshat - Personal\Visual Studio Code\Sensera\sensera-poc\model_dev_v1\health_users.csv"

# 12 epochs * 5 minutes = 1 Hour of temporal memory
SEQUENCE_LENGTH = 12 

def load_and_engineer_features(epochs_path, users_path):
    print("Loading and engineering features...")
    epochs_df = pd.read_csv(epochs_path)
    users_df = pd.read_csv(users_path)
    
    # Merge baselines and sort by user and time (CRITICAL for time-series)
    df = epochs_df.merge(users_df, on="user_id", how="left")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)

    # Calculate Deviations (Deltas)
    df["delta_hr"] = df["mean_hr"] - df["baseline_hr"]
    df["delta_spo2"] = df["mean_spo2"] - df["baseline_spo2"]
    df["delta_br"] = df["mean_br"] - df["baseline_br"]

    # Create Binary Target: 1 for Anomaly (Strain, Stress, Exercise), 0 for Normal/Sleep
    df["is_anomaly"] = df["true_latent_state"].apply(
        lambda x: 0 if x in ["normal", "sleep"] else 1
    )
    
    return df

def create_sequences(df, feature_cols, sequence_length):
    """
    Transforms 2D tabular data into 3D tensors: [samples, timesteps, features]
    We must ensure sequences don't cross over between different users.
    """
    print(f"Reshaping data into 3D sequences (Window Size: {sequence_length} epochs)...")
    
    # Scale features (Neural Networks need normalized inputs)
    scaler = StandardScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])
    
    X_list, y_list = [], []
    
    # Group by user so we don't accidentally create a sequence 
    # that starts with User A and ends with User B
    for user_id, user_df in df.groupby("user_id"):
        features = user_df[feature_cols].values
        labels = user_df["is_anomaly"].values
        
        # Sliding window approach
        for i in range(len(features) - sequence_length):
            X_list.append(features[i : i + sequence_length])
            # The label is the state at the END of the sequence
            y_list.append(labels[i + sequence_length - 1]) 
            
    return np.array(X_list), np.array(y_list)

def build_lstm_model(input_shape):
    """Builds a lightweight LSTM suitable for local GPU execution."""
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.3),
        LSTM(32, return_sequences=False),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid') # Binary classification output
    ])
    
    model.compile(
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

def main():
    # 1. Load Data
    try:
        df = load_and_engineer_features(EPOCHS_PATH, USERS_PATH)
    except FileNotFoundError:
        print("Error: CSV files not found. Check the file paths.")
        return

    feature_cols = [
        "mean_hr", "delta_hr", "max_hr",
        "mean_spo2", "delta_spo2", "min_spo2",
        "mean_br", "delta_br", "hr_br_ratio"
    ]

    # 2. Create 3D Tensors
    X, y = create_sequences(df, feature_cols, SEQUENCE_LENGTH)
    print(f"Tensor Shape: X={X.shape}, y={y.shape}")

    # 3. Train-Test Split (Chronological, no shuffling to respect time)
    # We take the last 20% of sequences as testing data
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    # 4. Build and Train LSTM
    print("\nInitializing LSTM Architecture...")
    model = build_lstm_model(input_shape=(SEQUENCE_LENGTH, len(feature_cols)))
    model.summary()

    print("\nTraining Model (Watch for GPU acceleration)...")
    # Early stopping prevents overfitting
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=15,
        batch_size=64, # Adjust based on your GPU VRAM
        callbacks=[early_stop],
        verbose=1
    )

    # 5. Evaluation
    print("\nEvaluating Sequence Model on Test Data...")
    y_pred_probs = model.predict(X_test)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()

    # 6. saving the model
    model.save(r'C:\Users\Akshat - Personal\Visual Studio Code\Sensera\sensera-poc\model_dev_v1\lstm_anomaly_model.keras')

    print("\n" + "="*50)
    print("LSTM CLASSIFICATION REPORT")
    print("="*50)
    print(classification_report(y_test, y_pred, target_names=["Normal", "Anomaly"]))

if __name__ == "__main__":
    # If running in a Jupyter Notebook, simply execute the cell.
    # GPU detection check:
    physical_devices = tf.config.list_physical_devices('GPU')
    if len(physical_devices) > 0:
        print(f"GPU Detected: {physical_devices[0]}")
    else:
        print("No GPU detected. Training will fall back to CPU.")
        
    main()

GPU Detected: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Loading and engineering features...
Reshaping data into 3D sequences (Window Size: 12 epochs)...
Tensor Shape: X=(20040, 12, 9), y=(20040,)

Initializing LSTM Architecture...
Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_6 (LSTM)               (None, 12, 64)            18944     
                                                                 
 dropout_6 (Dropout)         (None, 12, 64)            0         
                                                                 
 lstm_7 (LSTM)               (None, 32)                12416     
                                                                 
 dropout_7 (Dropout)         (None, 32)                0         
                                                                 
 dense_6 (Dense)             (None, 16)                528      